# A Simple Symbolic Calculator

F# counterpart of `Python/Chapter-10/Symbolic-Calculator.ipynb`. Run this notebook from top to bottom with a fresh .NET (F#) kernel.  The scanner and the parser are generated by the F# port of `Larkita` from the grammar `symbolic.lark`.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open LarkitaAst

## Grammar and typed syntax

The string literals of an alternative form the head of the abstract syntax tree:  `x := 2;` yields
`((':=', ';'), ('ID', 'x'), ('NUMBER', '2'))` and `x + 2*y;` yields `(';', ('+', ('ID', 'x'), ('*', ...)))`.  The
inner alternatives `("+" | "-")` and `("*" | "/")` abbreviate one alternative per operator.  A number is a
non-negative decimal without leading zeros, optionally with a fraction and an exponent.  The function `toStatement`
converts the trees into the closed types `Expression` and `Statement`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "symbolic.lark")

In [ ]:
/// A closed syntax tree makes every interpreter case visible to the compiler.
type Expression =
    | Number of float
    | Name of string
    | Unary of string * Expression
    | Binary of string * Expression * Expression

type Statement =
    | Assign of string * Expression
    | Evaluate of Expression

In [ ]:
let symbolicParser = withConflictWarnings Raise (fun () -> Larkita.Open("symbolic.lark", outFile = "symbolic.out"))

let rec toExpression (t: Ast) : Expression =
    match t with
    | Tok("NUMBER", digits) -> Number(System.Double.Parse(digits, System.Globalization.CultureInfo.InvariantCulture))
    | Tok("ID", name) -> Name name
    | Node([ operator ], [ left; right ]) -> Binary(operator, toExpression left, toExpression right)
    | _ -> failwithf "Unexpected tuple %s" (toString t)

let toStatement (t: Ast) : Statement =
    match t with
    | Node([ ":="; ";" ], [ Tok("ID", name); expression ]) -> Assign(name, toExpression expression)
    | Node([ ";" ], [ expression ]) -> Evaluate(toExpression expression)
    | _ -> failwithf "Unexpected tuple %s" (toString t)

module Parse =
    let symbolic text = toStatement (parse symbolicParser text)

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "symbolic.out")

## Evaluate expressions

The parser returns syntax; the evaluator owns variable lookup and arithmetic. Undefined names retain the original `NaN` semantics. A statement returns a new immutable environment plus an optional displayed value.

In [ ]:
open System

let rec evaluate (values: Map<string, float>) expression =
    match expression with
    | Number value -> value
    | Name name -> Map.tryFind name values |> Option.defaultValue Double.NaN
    | Binary (operator, left, right) ->
        let x, y = evaluate values left, evaluate values right
        match operator with
        | "+" -> x + y
        | "-" -> x - y
        | "*" -> x * y
        | "/" when y = 0.0 -> raise (DivideByZeroException "Division by zero")
        | "/" -> x / y
        | _ -> invalidArg "expression" operator
    | other -> invalidArg "expression" (sprintf "%A" other)

let execute values statement =
    match statement with
    | Assign (name, expression) -> Map.add name (evaluate values expression) values, None
    | Evaluate expression -> values, Some (evaluate values expression)

let calculate values text = Parse.symbolic text |> execute values

## Floating-point values

.NET supports IEEE floating-point values, including `NaN` and positive infinity. Arithmetic propagates an undefined value rather than inventing a variable value.

In [ ]:
Double.NaN, Double.PositiveInfinity, Double.PositiveInfinity - Double.PositiveInfinity
checkNotebook (Double.IsNaN (Double.PositiveInfinity - Double.PositiveInfinity))
checkNotebook (1.0 / Double.PositiveInfinity = 0.0)

## A reproducible session

Each line ends with a semicolon. Unlike an input loop, these cells can run unattended; the optional console loop below uses the same `calculate` function.

In [ ]:
let session = ["x := 2;"; "y := 3.5;"; "x + 2*y;"; "undefined;"; "zero := 0.25e2;"]
let values, outputs =
    session |> List.fold (fun (environment, outputs) text ->
        let next, value = calculate environment text
        next, outputs @ [value]) (Map.empty, [])
checkNotebook (values["x"] = 2.0 && values["y"] = 3.5 && values["zero"] = 25.0)
checkNotebook (outputs[2] = Some 9.0)
checkNotebook (outputs[3] |> Option.exists Double.IsNaN)
values, outputs

In [ ]:
let consoleSession () =
    let rec loop values =
        printf "calc> "
        match Console.ReadLine() with
        | null | "" -> values
        | text ->
            try
                let next, result = calculate values text
                result |> Option.iter (printfn "%g")
                loop next
            with error ->
                printfn "%s" error.Message
                loop values
    loop Map.empty
// Call consoleSession() explicitly in F# Interactive to read from the console.

In [ ]:
let invalid = try calculate Map.empty "1 + ;" |> ignore; false with _ -> true
checkNotebook invalid
for source in ["x<y;"; "x>y;"; "007;"] do
    let rejected = try calculate Map.empty source |> ignore; false with _ -> true
    checkNotebook rejected

let divisionRejected = try calculate Map.empty "1/0;" |> ignore; false with :? DivideByZeroException -> true
checkNotebook divisionRejected
